# ✅ ΟΛΟΚΛΗΡΩΘΗΚΕ — μην το ξανατρέξεις

Τέσσερα μοντέλα έγιναν εδώ (Qwen2.5-7B, Qwen3-4B, gemma-2-9b, Llama-3.1-8B):
15 από τα 16 κελιά δεν άλλαξαν, και το RQ2 μένει 4+/0− και με τα δύο prompts.

Το **gemma-2-2b**, που έλειπε, τρέχει στο **βήμα 1**,
`kaggle_g2b_triad.ipynb`, μαζί με αστέρα και clique στην ίδια συνεδρία. Το
κελί `MODELS` παρακάτω σταματά σκόπιμα, ώστε ένα λάθος «Run All» να μην
ξοδέψει ώρες GPU σε runs που υπάρχουν ήδη.

---

# Ablation — το διορθωμένο prompt στον δακτύλιο

**Τι διορθώνει.** Το cheap-talk system prompt είχε μία σταθερή πρόταση για τη
δρομολόγηση των μηνυμάτων, γραμμένη όταν ο αστέρας ήταν η μόνη τοπολογία:
*«ο κεντρικός agent εκπέμπει σε όλους τους περιφερειακούς…»*. Δεν είχε
placeholder, οπότε **500 runs του δακτυλίου** (cycle × cheap_talk) πήραν σωστή
δομική περιγραφή δακτυλίου και από κάτω περιγραφή αστέρα.

**Τι ΔΕΝ ήταν λάθος.** Η παράδοση των μηνυμάτων υπολογίζεται από την τοπολογία,
όχι από το prompt: ελέγχθηκε σε **64.640 / 64.640** agent-rounds. Κάθε agent του
δακτυλίου είδε ακριβώς 2 μηνύματα. Τα δεδομένα είναι πραγματικός δακτύλιος.

**Τι αποφασίζει αυτό το run.** Τρεις από τις τέσσερις κυψέλες του RQ2 είναι
μολυσμένες. Το RQ2 είναι ήδη ασθενές (sign test p ≈ 0,19) και δεν αντέχει και
δεύτερη επιφύλαξη. Εδώ ξανατρέχουμε αυτές τις κυψέλες με το διορθωμένο prompt.

| έκβαση | συμπέρασμα |
|---|---|
| ίδια νούμερα | το RQ2 γράφεται κανονικά, η επιφύλαξη φεύγει |
| αλλάζουν | το prompt οδηγούσε το αποτέλεσμα — σοβαρό εύρημα από μόνο του |
| αλλάζει μόνο το gemma-2-2b | η μοναδική αντιστροφή ήταν τεχνούργημα |

Και οι τρεις είναι πληροφοριακές.

**Τι δεν αλλάζει.** Ίδιο παιχνίδι, ίδιες αποδόσεις, 16 γύροι, κρυφός ορίζοντας,
μνήμη 10, T=0,7, μηνύματα ως 20 λέξεις, ίδιο `max_tokens` ανά μοντέλο. Αλλάζει
**μόνο μία παράγραφος του prompt**.


## Setup

1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Add-ons → Secrets → `HF_TOKEN`, **Attach to notebook**
   *(χρειάζεται και για τα δύο μοντέλα που απομένουν)*

Μετά: **Save Version → Save & Run All**.

**Απομένουν δύο μοντέλα, ~8 ώρες, μία συνεδρία.** Τα τρία πρώτα έχουν γίνει και
είναι σημειωμένα στο κελί `MODELS`· μην τα ξανατρέξεις.

⚠️ Αν η συνεδρία κοπεί στο τέλος, **μην υποθέσεις ότι χάθηκαν τα δεδομένα**:
στο gemma-2-9b κόπηκε το πακετάρισμα ενώ και τα 25 runs ήταν έτοιμα. Κοίτα το
Output panel για τον φάκελο `results/<model>_cycle_commfix`, όχι μόνο για zip.


In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), 'GPU off -- Settings -> Accelerator -> GPU T4 x2'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
GITHUB_REPO = 'https://github.com/stsimpe/cheaptalk_bench.git'
REPO_DIR = '/kaggle/working/repo'

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)

# Το repo root ΕΙΝΑΙ το package: campaign.py κάθεται στην κορυφή του clone.
assert os.path.exists('campaign.py'), f'campaign.py δεν είναι στο {os.getcwd()}'

# Χωρίς αυτή τη σημαία το notebook θα έτρεχε ΤΟ ΙΔΙΟ λάθος prompt που ήρθε να
# διορθώσει, και θα φαινόταν επιτυχημένο. Έχει ξανασυμβεί με άλλη σημαία.
for f in ('campaign.py', 'run_all_scenarios.py'):
    assert '--topology-aware-comm-prompt' in open(f, encoding='utf-8').read(), (
        f'Το {f} δεν έχει --topology-aware-comm-prompt -- κάνε git push πρώτα.')

print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'],
                              capture_output=True, text=True).stdout.strip())


## Ο έλεγχος που κρίνει το πείραμα

Δεν αρκεί να υπάρχει η σημαία. Πρέπει να αλλάζει **όντως** το prompt, και μόνο
αυτό. Το κελί παρακάτω χτίζει και τα δύο prompt για έναν agent δακτυλίου και τα
συγκρίνει. Κοστίζει μηδέν και δεν φορτώνει μοντέλο.


In [ ]:
import sys
sys.path.insert(0, '.')
from games import GAMES
from topology import make_topology
from prompts import build_system_prompt

topo = make_topology('cycle', 4)
common = dict(game=GAMES['pd'], condition='cheap_talk', n_neighbors=2,
              total_agents=4, topology_text=topo.describe(0))

old = build_system_prompt(**common)
new = build_system_prompt(**common, communication_text=topo.describe_communication(0))

assert 'central agent' in old, 'το παλιό prompt δεν έχει το κείμενο του αστέρα;'
assert 'central agent' not in new and 'peripheral' not in new, \
    'το διορθωμένο prompt ΕΧΕΙ ακόμα κείμενο αστέρα -- σταμάτα'
assert 'ring' in new and 'adjacent to it in the ring' in new, \
    'το διορθωμένο prompt δεν περιγράφει δακτύλιο -- σταμάτα'

diff = [l for l in new.splitlines() if l not in old.splitlines()]
print('η ΜΟΝΗ γραμμή που αλλάζει:\n')
for l in diff:
    print(' ', l.strip()[:200])
print(f'\nγραμμές που άλλαξαν: {len(diff)} (πρέπει να είναι 1)')
assert len(diff) == 1, 'άλλαξαν περισσότερα από τη μία παράγραφο -- σταμάτα'


In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token φορτώθηκε')
except Exception as e:
    print('Χωρίς HF_TOKEN (εντάξει μόνο για Qwen):', e)


## Τι θα τρέξει

Ανά μοντέλο, **cycle, PD μόνο, n=5**:

| σενάριο | κυψέλες | runs | κλήσεις | γιατί |
|---|---|---|---|---|
| `silence` | 1 | 5 | 320 | κυψέλη του RQ2, μολυσμένη |
| `no_sense` | 1 | 5 | 320 | κυψέλη του RQ2, μολυσμένη |
| `counterfactual` | 1 | 5 | 640 | κυψέλη του RQ2, μολυσμένη |
| `baseline` | 2 | 10 | 960 | το `cheap_talk` σκέλος είναι η πιο πολυχρησιμοποιημένη κυψέλη του corpus |

**25 runs, 2.240 κλήσεις ανά μοντέλο — άρα 50 runs σε αυτή τη συνεδρία.**

Το `baseline` τρέχει και τα δύο σκέλη του, και το `no_comm` σκέλος **δεν είχε
ποτέ το λάθος κείμενο**. Δεν είναι σπατάλη: τρέχει με πανομοιότυπο prompt με
πριν, άρα μετρά **πόσο μετακινείται μια κυψέλη από σκέτο θόρυβο δειγματοληψίας**.
Στα τρία μοντέλα που έγιναν, αυτός ο μάρτυρας κινήθηκε +0,131, +0,008 και
+0,003 — δηλαδή ο θόρυβος δεν είναι ο ίδιος για κάθε μοντέλο, και γι' αυτό
μετριέται χωριστά στο καθένα.


In [ ]:
raise SystemExit(
    'Αυτό το notebook έχει ολοκληρωθεί. Το gemma-2-2b που έλειπε τρέχει στο '
    'kaggle_g2b_triad.ipynb (βήμα 1). Οι τέσσερις φάκελοι commfix είναι ήδη '
    'κατεβασμένοι -- μην τους ξανατρέξεις.')

# ΑΠΟΜΕΝΟΥΝ ΔΥΟ ΜΟΝΤΕΛΑ, ΚΑΙ ΧΩΡΑΝΕ ΣΕ ΜΙΑ ΣΥΝΕΔΡΙΑ (~8 ώρες).
#
# Είναι και τα δύο που κρίνουν το RQ2: το gemma-2-2b είναι η μοναδική
# αντιστροφή ολόκληρης της καμπάνιας (αστέρας 0,67 - δακτύλιος 0,18) και το
# Llama έχει το μεγαλύτερο άνοιγμα στο silence (0,33 - 0,74).
MODELS = ['google/gemma-2-2b-it',              # ~1,9 h (ποτέ μετρημένο, εκτίμηση)
          'meta-llama/Llama-3.1-8B-Instruct']  # ~5,8 h   -> σύνολο ~8 h

# ΕΓΙΝΑΝ ΗΔΗ -- μην τα ξανατρέξεις, τα δεδομένα είναι κατεβασμένα:
#   Qwen/Qwen2.5-7B-Instruct   μάρτυρας +0,131, silence 0,394 -> 0,722 (p=0,034)
#   Qwen/Qwen3-4B              μάρτυρας +0,008, κανένα κελί δεν κινήθηκε
#   google/gemma-2-9b-it       μάρτυρας +0,003, κανένα κελί δεν κινήθηκε

SESSION   = 'A'
TOPOLOGY  = 'cycle'
SCENARIOS = ['silence', 'no_sense', 'counterfactual', 'baseline']
GAMES     = ['pd']

# Σταμάτα πριν ξεκινήσεις μοντέλο που δεν προλαβαίνει, αντί να σε κόψει το
# Kaggle στη μέση και να χαθεί η μισή δουλειά του.
WALL_HOURS = 10.5

def args_for(model):
    return ['--model', model, '--session', SESSION, '--topology', TOPOLOGY,
            '--scenarios', *SCENARIOS, '--games', *GAMES,
            '--topology-aware-comm-prompt']

print(f'{len(MODELS)} μοντέλα, {TOPOLOGY}, {GAMES}, διορθωμένο prompt')
for m in MODELS:
    print('  ', m)

## Preview — κοστίζει μηδέν

Για κάθε μοντέλο πρέπει να δεις:

- `PROMPT : topology-aware communication paragraph`
- `expecting : 25 run files`
- `games : ['pd']`
- `results -> : .../<model>_cycle_commfix`

Αν λείπει η γραμμή PROMPT ή ο φάκελος δεν τελειώνει σε `_commfix`, **σταμάτα**:
θα έτρεχες το παλιό prompt και θα μόλυνες και τα νέα δεδομένα.


In [ ]:
import subprocess, sys

ok = True
for m in MODELS:
    print('=' * 70)
    r = subprocess.run([sys.executable, 'campaign.py', *args_for(m), '--dry-run'],
                       capture_output=True, text=True)
    out = r.stdout.strip()
    print(out[:1000])
    if r.returncode != 0:
        ok = False
        print('ΣΦΑΛΜΑ:', r.stderr.strip()[-300:])
    for needle in ('topology-aware', '_commfix', 'expecting    : 25'):
        if needle not in out:
            ok = False
            print(f'ΛΕΙΠΕΙ ΑΠΟ ΤΟ ΠΛΑΝΟ: {needle!r}')

assert ok, 'Κάποιο plan απέτυχε -- μη συνεχίσεις.'
print('\n' + '=' * 70)
print('όλα τα πλάνα εντάξει')


## Εκτέλεση

Ένα μοντέλο τη φορά. Αν κάποιο αποτύχει, η σειρά συνεχίζει. Κάθε μοντέλο
ζιπάρεται μόλις τελειώσει, οπότε ακόμα κι αν κοπεί η συνεδρία, ό,τι έχει
τελειώσει είναι στο Output panel.


In [ ]:
import subprocess, sys, time

t0 = time.time()
results = []

for m in MODELS:
    elapsed_h = (time.time() - t0) / 3600
    if elapsed_h > WALL_HOURS:
        print(f'\n[STOP] {elapsed_h:.1f} h -- δεν ξεκινάω το {m}, δεν προλαβαίνει.')
        results.append((m, 'skipped', 0.0))
        continue

    print('\n' + '=' * 70)
    print(f'{m}   ({elapsed_h:.1f} h μέχρι τώρα)')
    print('=' * 70, flush=True)

    t = time.time()
    r = subprocess.run([sys.executable, 'campaign.py', *args_for(m)])
    mins = (time.time() - t) / 60
    status = 'OK' if r.returncode == 0 else f'FAILED (exit {r.returncode})'
    results.append((m, status, mins))
    print(f'\n--> {m}: {status}, {mins:.0f} λεπτά', flush=True)

print('\n' + '=' * 70)
print('ΑΠΟΛΟΓΙΣΜΟΣ')
for m, s, mins in results:
    print(f'  {s:22s} {mins:6.0f} λ   {m}')
print(f'\nσύνολο {(time.time() - t0) / 3600:.1f} h')


## Έλεγχος ότι τα δεδομένα είναι όντως τα νέα

Διαβάζει ένα αρχείο από κάθε μοντέλο και επιβεβαιώνει ότι η ρύθμιση γράφτηκε
μέσα στο record. Αν αυτό δείξει `False`, τα runs είναι με το παλιό prompt.


In [ ]:
import glob, json

for d in sorted(glob.glob('/kaggle/working/results/*_commfix')):
    files = glob.glob(f'{d}/**/*.json', recursive=True)
    if not files:
        print(f'{os.path.basename(d):40s} ΚΑΝΕΝΑ ΑΡΧΕΙΟ')
        continue
    rec = json.load(open(files[0], encoding='utf-8'))
    cfg = rec['config']
    print(f'{os.path.basename(d):40s} runs={len(files):3d}  '
          f'topology_aware={cfg.get("topology_aware_comm_prompt")}  '
          f'topology={rec["topology"]["type"]}  game={cfg["game"]}')


## Μάζεμα


In [ ]:
import glob, os, shutil

zips = sorted(glob.glob('/kaggle/working/*commfix*.zip'))
print(f'{len(zips)} zip ανά μοντέλο:')
for z in zips:
    print(f'   {os.path.getsize(z)/1e6:6.1f} MB  {os.path.basename(z)}')

if zips:
    box = '/kaggle/working/commfix_all'
    os.makedirs(box, exist_ok=True)
    for z in zips:
        shutil.copy(z, box)
    out = shutil.make_archive('/kaggle/working/commfix_runs_all', 'zip', box)
    print(f'\nκατέβασε αυτό: {out}  ({os.path.getsize(out)/1e6:.1f} MB)')
else:
    print('\nΚΑΝΕΝΑ ZIP -- δες τον απολογισμό παραπάνω.')


## Μετά

Κατέβασε το `commfix_runs_all.zip` στο `diplomatikh/rq4/commfix_cycle/`.
**Μην το βάλεις στους δέκα φακέλους του πλέγματος** — είναι δεύτερη γενιά
prompt και δεν πρέπει να μπει ποτέ στο ίδιο κελί με τα παλιά.

Η σύγκριση, ανά μοντέλο, cycle μείον star στο PD:

| κυψέλη | παλιό prompt | νέο prompt |
|---|---|---|
| `no_comm` | ίδιο και στα δύο (ποτέ δεν είχε το λάθος κείμενο) | μέτρο θορύβου |
| `silence` | γνωστό | ; |
| `no_sense` | γνωστό | ; |
| `counterfactual` | γνωστό | ; |
| `baseline_cheap_talk` | γνωστό | ; |

Το κατώφλι δεν είναι «άλλαξε κάτι;» αλλά **«άλλαξε περισσότερο από όσο
μετακινήθηκε το `no_comm`, που έτρεξε με πανομοιότυπο prompt;»**

Μένουν σκόπιμα με το παλιό prompt και τεκμηριώνονται ως περιορισμός: τα SH
σκέλη του δακτυλίου και οι έξι κυψέλες framing/context του δακτυλίου. Στηρίζουν
ισχυρισμούς της μορφής *δακτύλιος ≈ αστέρας*, που ένα confound μόνο στον
δακτύλιο θα τους χαλούσε, όχι θα τους δημιουργούσε.
